# T5 Fine-tuning for Text Simplification

Fine-tunes **T5-small** to simplify complex text into plain language.

### Dataset
We use **`bogdancazan/wikilarge-text-simplification`** (36MB, 150k pairs) — much smaller and faster to download than the GEM datasets that time out.
Column names: `Normal` (complex) and `Simple` (simplified).

### Key design choice: data filtering
Raw WikiLarge pairs are often near-identical (source ≈ target), which causes the model to learn to **copy** instead of simplify. We filter before training to keep only pairs where real simplification happened (< 80% word overlap, target shorter than source).

**Before running:** Go to `Runtime > Change runtime type > T4 GPU`

**Output:** A fine-tuned model (~250MB zip) you download and drop into the project.

## Step 1 — Install dependencies

- `transformers` — HuggingFace library with T5 model + Trainer
- `datasets` — loads the dataset from HuggingFace Hub
- `sentencepiece` — tokenizer backend that T5 uses internally
- `accelerate` — required by Trainer for GPU training

In [ ]:
!pip install -q transformers datasets sentencepiece accelerate

## Step 2 — Verify GPU

Training on CPU would take 8+ hours. A T4 GPU does it in ~25 minutes.
If this says "No GPU", go to `Runtime > Change runtime type > T4 GPU` and re-run.

In [ ]:
import torch

if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('NO GPU! Go to Runtime > Change runtime type > T4 GPU')

## Step 3 — Load the dataset

**bogdancazan/wikilarge-text-simplification** — 150k (complex, simple) sentence pairs, 36MB total.
Downloads fast. Columns are `Normal` (complex) and `Simple` (simplified).

We'll look at a few examples to understand what we're working with.

In [ ]:
from datasets import load_dataset

dataset = load_dataset('bogdancazan/wikilarge-text-simplification')
print('Dataset loaded.')
print(f'Train: {len(dataset["train"]):,} pairs')
print(f'Val:   {len(dataset["validation"]):,} pairs')
print()

# Look at 3 examples
for i in [0, 100, 5000]:
    ex = dataset['train'][i]
    print(f'--- Example {i} ---')
    print(f'NORMAL: {ex["Normal"]}')
    print(f'SIMPLE: {ex["Simple"]}')
    same = ex['Normal'].strip().lower() == ex['Simple'].strip().lower()
    print(f'Identical? {"YES — this is the problem!" if same else "No"}')
    print()

## Step 4 — Measure the problem: how similar are Normal and Simple?

We compute **word overlap** — what fraction of words in `Normal` also appear in `Simple`.

- Overlap = 1.0 → every word appears in both (model will learn to copy)
- Overlap < 0.8 → at least 20% of words changed (real simplification)

If most pairs have high overlap, the model learns "just copy the input" as the safest strategy.

In [ ]:
import random
import statistics

def word_overlap(text1, text2):
    """Fraction of words in text1 that also appear in text2."""
    words1 = set(text1.lower().split())
    words2 = set(text2.lower().split())
    if not words1:
        return 1.0
    return len(words1 & words2) / len(words1)

# Sample 5000 pairs and measure overlap
random.seed(42)
indices = random.sample(range(len(dataset['train'])), 5000)

overlaps = []
identical_count = 0

for idx in indices:
    ex = dataset['train'][idx]
    src = ex['Normal'].strip()
    tgt = ex['Simple'].strip()
    if src.lower() == tgt.lower():
        identical_count += 1
    overlaps.append(word_overlap(src, tgt))

print(f'Analyzed 5,000 random pairs:')
print(f'  Completely identical:              {identical_count} ({identical_count/50:.1f}%)')
print(f'  Mean word overlap:                 {statistics.mean(overlaps):.1%}')
print(f'  Median word overlap:               {statistics.median(overlaps):.1%}')
print(f'  Pairs with <80% overlap (quality): {sum(1 for o in overlaps if o < 0.80)}')
print()
print('Pairs with <80% overlap are the ones worth training on.')

## Step 5 — Filter for quality training pairs

We keep a pair only if ALL of these are true:

| Rule | Why |
|------|-----|
| Normal ≠ Simple | Can't learn from identical pairs |
| Normal ≥ 8 words | Short sentences don't have much to simplify |
| Simple ≥ 3 words | Avoid degenerate/empty simplifications |
| Simple ≤ Normal length | Simplification should shorten, not expand |
| Word overlap < 80% | At least 20% of content must change |

This is **data curation** — in NLP research, cleaning data is often more impactful than tuning the model.

In [ ]:
def is_good_pair(example):
    """Returns True only if this pair shows meaningful simplification."""
    src = example['Normal'].strip()
    tgt = example['Simple'].strip()

    # Rule 1: not identical
    if src.lower() == tgt.lower():
        return False

    src_words = src.split()
    tgt_words = tgt.split()

    # Rule 2: source long enough to be worth simplifying
    if len(src_words) < 8:
        return False

    # Rule 3: target not degenerate
    if len(tgt_words) < 3:
        return False

    # Rule 4: target should be shorter (simplification = reduction)
    if len(tgt_words) > len(src_words):
        return False

    # Rule 5: at least 20% of content changed
    if word_overlap(src, tgt) >= 0.80:
        return False

    return True

print('Filtering training data...')
filtered_train = dataset['train'].filter(is_good_pair)

print('Filtering validation data...')
filtered_val = dataset['validation'].filter(is_good_pair)

print(f'\n--- Results ---')
print(f'Train: {len(dataset["train"]):,} → {len(filtered_train):,} ({len(filtered_train)/len(dataset["train"]):.1%} kept)')
print(f'Val:   {len(dataset["validation"]):,} → {len(filtered_val):,}')

if len(filtered_train) < 5000:
    print('\nWARNING: Very few pairs passed the filter.')
    print('Relax Rule 5: change >= 0.80 to >= 0.85 in is_good_pair() and re-run.')
else:
    print(f'\nGood — {len(filtered_train):,} quality training pairs.')

In [ ]:
# Verify: show 5 filtered pairs — these should show CLEAR simplification
print('=== Filtered examples (should show real simplification) ===\n')
for i in range(5):
    ex = filtered_train[i]
    ov = word_overlap(ex['Normal'], ex['Simple'])
    sw = len(ex['Normal'].split())
    tw = len(ex['Simple'].split())
    print(f'--- Pair {i+1}  |  overlap: {ov:.0%}  |  {sw} → {tw} words ---')
    print(f'NORMAL: {ex["Normal"]}')
    print(f'SIMPLE: {ex["Simple"]}')
    print()

In [ ]:
# Cap at 40k train / 2k val so training stays under 30 minutes
MAX_TRAIN = 40000
MAX_VAL = 2000

train_data = filtered_train.shuffle(seed=42).select(
    range(min(MAX_TRAIN, len(filtered_train)))
)
val_data = filtered_val.shuffle(seed=42).select(
    range(min(MAX_VAL, len(filtered_val)))
)

print(f'Final training set:   {len(train_data):,} pairs')
print(f'Final validation set: {len(val_data):,} pairs')

## Step 6 — Tokenize

T5 is a **text-to-text** model — every task is framed as "convert this text to that text." The task is specified by a **prefix**.

- We prepend `simplify: ` to every input so the model learns: "when you see this prefix, simplify."
- At inference time we use the same prefix.

We don't pad during tokenization — the `DataCollatorForSeq2Seq` in Step 7 pads each batch dynamically (faster).

In [ ]:
from transformers import T5Tokenizer

MODEL_NAME = 't5-small'
tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME)

PREFIX = 'simplify: '
MAX_INPUT_LEN = 128
MAX_TARGET_LEN = 128

def tokenize(batch):
    # Prepend task prefix to the complex (Normal) sentences
    inputs = [PREFIX + text for text in batch['Normal']]

    # Tokenize inputs — no padding, collator handles it
    model_inputs = tokenizer(
        inputs,
        max_length=MAX_INPUT_LEN,
        truncation=True,
    )

    # Tokenize targets (Simple sentences)
    labels = tokenizer(
        batch['Simple'],
        max_length=MAX_TARGET_LEN,
        truncation=True,
    )

    model_inputs['labels'] = labels['input_ids']
    return model_inputs

print('Tokenizing training data...')
tokenized_train = train_data.map(
    tokenize, batched=True, remove_columns=train_data.column_names
)

print('Tokenizing validation data...')
tokenized_val = val_data.map(
    tokenize, batched=True, remove_columns=val_data.column_names
)

print(f'Done. Train: {len(tokenized_train):,} | Val: {len(tokenized_val):,}')

## Step 7 — Set up model and trainer

| Setting | Value | Why |
|---------|-------|-----|
| `Seq2SeqTrainer` | (not plain Trainer) | Supports `predict_with_generate` for proper text generation during eval |
| `num_train_epochs` | 3 | Enough to learn; more risks overfitting |
| `batch_size` | 16 | Fits in T4's 15GB VRAM |
| `fp16` | True | Half-precision = ~2x faster, no quality loss |
| `load_best_model_at_end` | True | Auto-keeps the best checkpoint |
| `warmup_steps` | 500 | Gradually ramps learning rate to avoid instability |
| `DataCollatorForSeq2Seq` | — | Dynamic per-batch padding + sets label padding to -100 so loss ignores it |

In [ ]:
from transformers import (
    T5ForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq,
)

model = T5ForConditionalGeneration.from_pretrained(MODEL_NAME)

# Handles dynamic padding and label masking per batch
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

training_args = Seq2SeqTrainingArguments(
    output_dir='./t5-simplifier',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=500,
    weight_decay=0.01,
    logging_steps=200,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    predict_with_generate=True,
    generation_max_length=128,
    fp16=True,
    report_to='none',
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    processing_class=tokenizer,
    data_collator=data_collator,
)

params = sum(p.numel() for p in model.parameters())
print(f'Model:    {MODEL_NAME} ({params/1e6:.0f}M parameters)')
print(f'Training: {len(tokenized_train):,} samples × 3 epochs')
print(f'ETA:      ~20-30 minutes on T4')
print('Ready to train.')

## Step 8 — Train

Takes ~20-30 minutes on T4. Watch the numbers:
- **Training loss** should drop steadily (from ~2.0 toward ~1.0 or lower)
- **Validation loss** should also drop (if it rises, the model is overfitting)

The best checkpoint is automatically kept at the end.

In [ ]:
trainer.train()

## Step 9 — Test: does it actually simplify?

We test in **two ways**:

**Part A — Ground truth check:** Feed sentences from the validation set that the model has never seen. If the model changes these, it's genuinely simplifying (not just "these test sentences are out-of-domain").

**Part B — Custom sentences:** Our own test sentences. Note: the model was trained on Wikipedia article text. It works best on that style. Science textbook sentences ("The researchers hypothesized...") are a different register and may not change as much.

In [ ]:
model.eval()

def simplify(text):
    """Simplify one sentence using the fine-tuned model."""
    inputs = tokenizer(
        'simplify: ' + text,
        return_tensors='pt',
        max_length=128,
        truncation=True,
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    outputs = model.generate(
        inputs['input_ids'],
        attention_mask=inputs['attention_mask'],
        max_length=128,
        num_beams=4,
        length_penalty=0.8,
        no_repeat_ngram_size=3,
        early_stopping=True,
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)


# ── Part A: Ground truth check ──────────────────────────────────────────────
# Use 5 held-out validation examples the model never saw during training.
# We know the "correct" simplified version — compare model output to it.
print('=' * 65)
print('PART A — VALIDATION SET (in-domain, ground truth available)')
print('=' * 65)

for i in range(5):
    ex = val_data[i]                          # raw (unfiltered) val examples
    result = simplify(ex['Normal'])
    is_copy = result.strip().lower() == ex['Normal'].strip().lower()
    print(f'\nINPUT:    {ex["Normal"]}')
    print(f'MODEL:    {result}')
    print(f'EXPECTED: {ex["Simple"]}')
    print(f'          [{"COPIED" if is_copy else "CHANGED"}]')

# ── Part B: Custom sentences ─────────────────────────────────────────────────
print('\n\n' + '=' * 65)
print('PART B — CUSTOM SENTENCES (Wikipedia-style prose)')
print('The model was trained on Wikipedia text. These sentences match that style.')
print('=' * 65)

# These sentences are written in Wikipedia article style (biographical/encyclopedic)
# rather than science textbook style — closer to the training distribution.
wiki_style_sentences = [
    'He was born in the city of Vienna and later became one of the most influential composers of the classical period.',
    'The treaty was signed in 1648 and ended the Thirty Years War, which had devastated much of central Europe.',
    'The species is found primarily in tropical regions and feeds on a wide variety of insects and small mammals.',
    'She was elected to the position of prime minister in 1979 and served for three consecutive terms.',
    'The organization was established in 1945 with the primary objective of maintaining international peace and security.',
]

copy_count = 0
for s in wiki_style_sentences:
    result = simplify(s)
    is_copy = result.strip().lower() == s.strip().lower()
    if is_copy:
        copy_count += 1
    print(f'\nINPUT:  {s}')
    print(f'OUTPUT: {result}')
    print(f'        [{"COPIED — model did not change this" if is_copy else "CHANGED — good!"}]')

print('\n' + '=' * 65)
simplified = len(wiki_style_sentences) - copy_count
if simplified >= 3:
    print(f'SUCCESS: {simplified}/{len(wiki_style_sentences)} simplified. Model is working.')
elif simplified > 0:
    print(f'PARTIAL: {simplified}/{len(wiki_style_sentences)} simplified.')
    print('This is still useful — the model has learned some simplification patterns.')
else:
    print('FAIL: Nothing simplified. Check Part A results above.')
    print('If Part A shows changes, the model works but our test sentences are too far from Wikipedia style.')

## Step 10 — Save and download

If Step 9 looks good (model is simplifying, not copying), run these two cells.

In [ ]:
model.save_pretrained('./t5-simplifier')
tokenizer.save_pretrained('./t5-simplifier')
print('Model saved to ./t5-simplifier/')

In [ ]:
!zip -r t5-simplifier.zip t5-simplifier/

from google.colab import files
files.download('t5-simplifier.zip')
print('Download started (~250MB).')

## After downloading

1. Unzip `t5-simplifier.zip` into your project folder
2. In `simplify.py`, change the default model path from `t5-small` to `./t5-simplifier`
3. Change the prefix in `simplify_with_t5()` from `summarize:` to `simplify:`
4. Run normally — the fine-tuned model takes over

### What this demonstrates (for your report)
- **Data curation**: We didn't just dump all 150k pairs in — we filtered for quality. This is standard practice in NLP research.
- **Task-specific fine-tuning**: The `simplify:` prefix teaches T5 a new task beyond its pre-training.
- **Offline inference**: Once downloaded, the model runs entirely locally — no API calls, no internet required.